# Debug Order Service / Payment Status

This notebook runs the existing `payment_app/payment_app/test_order_request.py` script and inspects the output, including status code and response body. It also checks the payment status endpoint used by the order service.

In [ ]:
import json
import os
from pathlib import Path
from subprocess import PIPE, Popen

BASE_DIR = Path(r"e:\Learning Projects\Lumia_Backend_updated")
PAYMENT_APP_DIR = BASE_DIR / "payment_app" / "payment_app"
SCRIPT_PATH = PAYMENT_APP_DIR / "test_order_request.py"
OUTPUT_PATH = PAYMENT_APP_DIR / "test_order_response.json"

print("BASE_DIR:", BASE_DIR)
print("SCRIPT_PATH:", SCRIPT_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("SCRIPT exists:", SCRIPT_PATH.exists())
print("WORKING DIR exists:", PAYMENT_APP_DIR.exists())

## Run the test script and capture stdout/stderr

Execute the reproduction script in its working directory and display the output.

In [ ]:
process = Popen(
    ["python", str(SCRIPT_PATH)],
    cwd=str(PAYMENT_APP_DIR),
    stdout=PIPE,
    stderr=PIPE,
    text=True,
)
out, err = process.communicate(timeout=30)
print("Return code:", process.returncode)
print("STDOUT:\n", out)
print("STDERR:\n", err)

if OUTPUT_PATH.exists():
    with OUTPUT_PATH.open("r", encoding="utf-8") as fp:
        data = json.load(fp)
    print("\nCaptured JSON response:")
    print(json.dumps(data, indent=2))
else:
    print("No response file created.")

## Hit the payment status endpoint directly

This checks whether the test payment reference resolves successfully in the payment service.

In [ ]:
import urllib.request
import urllib.error

url = "http://127.0.0.1:8006/api/v1/payments/payref_4bd9817337982e29f26f1a60/status"
req = urllib.request.Request(url, headers={"Accept": "application/json"})
try:
    with urllib.request.urlopen(req, timeout=20) as resp:
        body = resp.read().decode("utf-8")
        print("Status:", resp.status)
        print("Body:\n", body)
except urllib.error.HTTPError as exc:
    print("HTTPError", exc.code)
    print(exc.read().decode("utf-8"))
except Exception as exc:
    print(type(exc).__name__, exc)
